# Running the experiments

This notebook shows the full experimental process for one architecture–dataset
pair. To run a different pair, change `MODEL` and `DATASET` in the settings
cell.

The notebook covers four stages:

1. Training the untuned reference configuration
2. Running DEHB and retraining its best configuration
3. Running Random Search and retraining its best configuration
4. Running a convergence probe to help choose the fidelity range

The reference, DEHB and Random Search experiments all use `trainer.train`, so
they share the same data pipeline, loss, metrics and random seed. The main
difference between them is the hyperparameter configuration being trained.


## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# NumPy is pinned for compatibility with torch-spatiotemporal.
# Colab may show dependency warnings after this installation.
# Restart the runtime after this cell before importing the project modules.
!pip install torch-geometric
!pip install torch-spatiotemporal dehb
!pip install "pytorch-lightning==2.6.5"
!pip install "numpy==1.26.4"

In [ ]:
import sys

CODE_DIR = '/content/drive/MyDrive/new_code'     # folder containing the project .py files
DATA_ROOT = '/content/drive/MyDrive/AutoSTGNN/data'
RESULTS_DIR = '/content/drive/MyDrive/AutoSTGNN/search_results'

if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)

import torch
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')

## 2. Settings for the pair

Random Search is given the same allocated epoch budget as DEHB. Its budget is
calculated as the number of trials multiplied by the fixed fidelity, using
DEHB's maximum fidelity. The fidelity ranges were chosen based on the
convergence probes, and DEHB uses `eta = 3`.


In [ ]:
MODEL = 'agcrn'        # 'agcrn' | 'stgcn' | 'graphwavenet'
DATASET = 'metrla'     # 'metrla' | 'pemsbay' | 'electricity'

# Fidelity range and Random Search trial count for each model-dataset pair
SETTINGS = {
    ('agcrn', 'metrla'):             (3, 11, 25),
    ('stgcn', 'metrla'):             (3, 11, 25),
    ('graphwavenet', 'metrla'):      (4, 12, 25),
    ('agcrn', 'pemsbay'):            (3, 11, 20),
    ('stgcn', 'pemsbay'):            (3, 11, 20),
    ('graphwavenet', 'pemsbay'):     (4, 12, 20),
    ('agcrn', 'electricity'):        (4, 12, 20),
    ('stgcn', 'electricity'):        (4, 12, 20),
    ('graphwavenet', 'electricity'): (4, 12, 20),
}

MIN_FID, MAX_FID, N_TRIALS = SETTINGS[(MODEL, DATASET)]
EPOCH_BUDGET = N_TRIALS * MAX_FID     # same allocated epoch budget for both methods
RETRAIN_EPOCHS = 30                   # maximum epochs for final retraining

print(MODEL, DATASET)
print('fidelity range', MIN_FID, 'to', MAX_FID)
print('random search:', N_TRIALS, 'trials at', MAX_FID, 'epochs')
print('allocated epoch budget:', EPOCH_BUDGET)

## 3. Reference configuration

The reference run uses the default hyperparameters defined in
`trainer.DEFAULT_MODEL_KWARGS`. This gives the untuned configuration used as
the comparison point for DEHB and Random Search.

The reference configurations were run directly through `trainer.train` with a
learning rate of 0.001, batch size 64 and a maximum of 30 epochs, matching the
final training setup used for the selected configurations. `train_baselines.py`
is the group's shared baseline script and is included in the project files for
completeness.


In [ ]:
from trainer import train

# Runtime varies depending on the model, dataset and GPU.
predictor, trainer_obj, test_results, best_ckpt, best_val_mae = train(
    dataset_name=DATASET,
    model_name=MODEL,
    window=12,
    horizon=12,
    batch_size=64,
    lr=0.001,
    max_epochs=RETRAIN_EPOCHS,
    base_root=DATA_ROOT,
    model_kwargs=None,          # uses the reference settings in DEFAULT_MODEL_KWARGS
    patience=30,
    seed=42,
)
print('reference test MAE:', test_results[0]['test_mae'])

## 4. DEHB

`run_dehb` carries out the search and then retrains the selected configuration.
The configuration with the lowest validation MAE is selected, so test metrics
are not used to choose the incumbent.

The trial log is saved throughout the search. If a Colab session disconnects,
the run can be continued by passing the saved log to `resume_from`.


In [ ]:
from dehb_runner import run_dehb

# The search can take several hours, so results are saved as it runs.
dehb_log = run_dehb(
    model_name=MODEL,
    dataset_name=DATASET,
    min_fidelity=MIN_FID,
    max_fidelity=MAX_FID,
    total_epoch_budget=EPOCH_BUDGET,
    retrain_epochs=RETRAIN_EPOCHS,
    base_root=DATA_ROOT,
    results_dir=RESULTS_DIR,
    seed=42,
)
print(len(dehb_log), 'trials')

### Resuming an interrupted DEHB search

DEHB searches can take several hours, so a Colab session may disconnect before
the search finishes. Because `run_dehb` saves the trial log after each trial,
an interrupted run can be continued by passing that log to `resume_from`.

DEHB also saves its internal state in the
`dehb_internal_<model>_<dataset>` directory under `results_dir`. This allows
the search to continue with the same population and bracket position instead
of starting again.

This is useful when a run stops partway through the search. If the allocated
epoch budget has already been completed, DEHB treats the search as finished
and returns without starting new trials. To deliberately start a new search,
the saved internal state must first be removed.


In [ ]:
import glob, json, os

# Load the most recent DEHB trial log.
DEHB_LOG = sorted(glob.glob(RESULTS_DIR + '/' + MODEL + '_' + DATASET + '_dehb_2*.json'))[-1]
done = json.load(open(DEHB_LOG))
used = sum(t.get('fidelity', 0) for t in done)
print(os.path.basename(DEHB_LOG), '|', len(done), 'trials |', used, 'of', EPOCH_BUDGET, 'epochs used')

# Continue the interrupted search using the same log.
dehb_log = run_dehb(
    model_name=MODEL,
    dataset_name=DATASET,
    min_fidelity=MIN_FID,
    max_fidelity=MAX_FID,
    total_epoch_budget=EPOCH_BUDGET,
    retrain_epochs=RETRAIN_EPOCHS,
    base_root=DATA_ROOT,
    results_dir=RESULTS_DIR,
    resume_from=DEHB_LOG,
    seed=42,
)

# Remove the saved DEHB state only when starting a completely new search:
# !rm -rf "{RESULTS_DIR}/dehb_internal_{MODEL}_{DATASET}"

## 5. Random Search

Random Search is run in two steps. First, `run_random_search` trains
`N_TRIALS` sampled configurations at the fixed fidelity. Then
`retrain_random_search` selects the configuration with the lowest validation
MAE and retrains it.

The final retraining uses the same `retrain_incumbent` function as DEHB, which
keeps the final training procedure consistent between the two methods.

`max_trials` limits incumbent selection to the trials included in the allocated
budget. This is useful if a saved log contains additional trials.


In [ ]:
from random_search import run_random_search

rs_log = run_random_search(
    model_name=MODEL,
    dataset_name=DATASET,
    n_trials=N_TRIALS,
    max_epochs=MAX_FID,
    base_root=DATA_ROOT,
    results_dir=RESULTS_DIR,
)
print(len(rs_log), 'trials')

### Resuming an interrupted Random Search

Resuming Random Search is simpler because it does not have an internal
optimiser state. All configurations are sampled from the seeded search space
at the start of the run. When an existing log is supplied, completed trials
are skipped and the run continues from the next configuration in the same
sequence.


In [ ]:
import glob, json, os
RS_LOG_PARTIAL = sorted(glob.glob(RESULTS_DIR + '/' + MODEL + '_' + DATASET + '_RS_2*.json'))[-1]
print('resuming from', os.path.basename(RS_LOG_PARTIAL),
      'with', len(json.load(open(RS_LOG_PARTIAL))), 'trials done')

rs_log = run_random_search(
    model_name=MODEL,
    dataset_name=DATASET,
    n_trials=N_TRIALS,
    max_epochs=MAX_FID,
    base_root=DATA_ROOT,
    results_dir=RESULTS_DIR,
    resume_from=RS_LOG_PARTIAL,
)

In [ ]:
import glob
from retrain_rs import retrain_random_search

# Use the latest Random Search log, or replace this path with an existing one.
RS_LOG = sorted(glob.glob(RESULTS_DIR + '/' + MODEL + '_' + DATASET + '_RS_2*.json'))[-1]
print('using', RS_LOG)

rs_incumbent = retrain_random_search(
    results_path=RS_LOG,
    model_name=MODEL,
    dataset_name=DATASET,
    base_root=DATA_ROOT,
    results_dir=RESULTS_DIR,
    retrain_epochs=RETRAIN_EPOCHS,
    max_trials=N_TRIALS,
    fidelity=MAX_FID,
)
rs_incumbent['test_mae']

## 6. Convergence probe

The convergence probe trains five sampled configurations for a short run and
tracks training and validation MAE across epochs. The convergence behaviour
was used to guide the choice of fidelity ranges used in the search experiments.

Results are saved under `<output_root>/<model>/`. Because the output path does
not include the dataset name, a separate `output_root` is used for each
dataset.


In [ ]:
from convergence import run_convergence_analysis

probe = run_convergence_analysis(
    model_name=MODEL,
    dataset_name=DATASET,
    n_configs=5,
    max_epochs=15,
    base_root=DATA_ROOT,
    output_root='/content/drive/MyDrive/AutoSTGNN/convergence_plots/' + DATASET,
    seed=42,
)

## 7. Comparing the three

The Random Search and DEHB incumbent retrains save their results to JSON and
Excel files, which are read back below. The reference run in Section 3 prints
its test MAE directly instead of saving a separate result record in this
notebook.

Together, these three values give the results for this architecture–dataset
pair used in the final comparison.


In [ ]:
import json, glob

def latest(pattern):
    hits = sorted(glob.glob(RESULTS_DIR + '/' + pattern))
    if not hits:
        return None
    r = json.load(open(hits[-1]))
    return r[0] if isinstance(r, list) else r

rows = [
    # The reference MAE is printed in Section 3 rather than saved here.
    ('Random search', latest(MODEL + '_' + DATASET + '_random_search_incumbent_*.json')),
    ('DEHB',          latest(MODEL + '_' + DATASET + '_dehb_incumbent_*.json')),
]

for name, r in rows:
    if r is None:
        print(name.ljust(15), 'no file')
    else:
        print(name.ljust(15), 'test MAE', round(r['test_mae'], 4))